## ChatGPT -- fix a few question typos in dataset

Ask VQA to ChatGPT.

Docs:

* how to upload images: https://platform.openai.com/docs/guides/vision
* types of models: https://stackoverflow.com/questions/75774873/openai-api-error-this-is-a-chat-model-and-not-supported-in-the-v1-completions
* token limits: https://platform.openai.com/settings/organization/limits
* models: https://platform.openai.com/docs/models/model-endpoint-compatibility

In [9]:
dir_api = '/Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/LMM_outputs/chatgpt_api/' #store API results for example data
dir_api_old = '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/chatgpt_api/' # combine and re-write with old data
model = "gpt-5-nano-2025-08-07"
temperature = 1.0 ## only allowed


redo_file = '/Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/question_updates.json'
key_file = '/Users/jnaiman/.openai/key.txt'

jsons_dir = '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/jsons/' # directory where jsons created with figure are stored
imgs_dir = '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/imgs/' # where images are stored

# for saving temp images for reading in
tmp_dir = '/Users/jnaiman/Downloads/tmp/'

img_format = 'png'

In [10]:
import openai
import base64
from openai import OpenAI
from PIL import Image
import numpy as np
import json
import re
import pickle
import os
from glob import glob

# debug
from importlib import reload

from sys import path
path.append('../')
import utils.llm_utils
reload(utils.llm_utils)
from utils.llm_utils import parse_qa, load_image, get_img_json_pair, parse_for_errors

from utils.plot_qa_utils import get_nplots
from copy import deepcopy

In [11]:
def send_to_chatgpt(question_list, client, image_path, encoded_image,
                    model ="gpt-4o-mini", 
                    tmp_dir = '/Users/jnaiman/Downloads/tmp/',
                    test_run = True, fac=1.0, img_format='png',
                    verbose=True, temperature=1.0,
                    subset_questions_by_keys = None):
    """
    subset_questions_by_keys : only use a subset of questions to ping to model, e.g. ['median', 'how many gaussians']
    """

    iFac = 2.0 # just in case we want to progressively make the image smaller
    success = False
    is_subset = False
    new_fac = fac
    while not success:
        try:
        #if True:
            # current question format is: ['persona', 'context','question', 'format'] (see readme in example_hists)
            question = question_list['context'] + " " + question_list['question'] + " " + question_list['format']
            # lowercase the first word, just in case
            question = question.lstrip() # no whitespace
            question = question[0].lower() + question[1:]
            if verbose: print('   on question:',question)
            # Prepare the API request
            # prompt = f"I am going to show you an image. Here is the image: [Image: {encoded_image}]. Now, {question}" # this puts it in twice!
            # prompt_save = f"I am going to show you an image. Here is the image: [Image: <ENCODED IMAGE>]. Now, {question}"
            prompt = f"I am going to show you an image. Now, {question}"
            prompt_save = f"I am going to show you an image. Now, {question}"

            if subset_questions_by_keys is not None and type(subset_questions_by_keys) == type([]): # make sure list
                for s in subset_questions_by_keys:
                    if s in question:
                        is_subset = True
            else:
                is_subset = True
            
            if not test_run and is_subset:
                # Send the request to the GPT-4o API
                response = client.chat.completions.create(
                    model = model,
                    messages=[
                        {"role": "system", "content": question_list['persona']},
                        {"role":"user", "content": [
                            {
                            "type": "text",
                            "text": prompt
                            },
                            {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/{img_format};base64,{encoded_image}"
                            }
                            }
                        ]
                        }
                    ],
                    temperature=temperature
                )
                success = True
            elif not is_subset:
                response = 'Not asked'
                success = True
            else:
                success = True
        except Exception as e:
        #else:
            print('[ERROR]', str(e))
            new_fac = fac/iFac
            print('new fac = ', new_fac)
            encoded_image = load_image(image_path,fac=new_fac, tmp_dir=tmp_dir)
            iFac += 1
    
    if not test_run and is_subset:
        # Get the response from the API
        answer = response.choices[0].message.content
        question_list['raw answer'] = answer
        # format answer
        answer_format = answer.replace("```json\n",'').replace("\n```",'')
        try:
            question_list['Response'] = json.loads(answer_format)
        except:
            question_list['Response'] = answer_format
            question_list['Error'] = 'JSON formatting'
        question_list['Response String'] = answer_format
        success = True
    elif not is_subset:
        answer = response
        question_list['raw answer'] = answer
        question_list['Response'] = answer
        question_list['Response String'] = answer
        success = True
    else:
        question_list['Response'] = 'TEST RUN'
        question_list['Response String'] = 'TEST RUN'
    question_list['fac'] = new_fac

    return question_list, prompt_save

In [12]:
def print_qa(pickle_file, qa_in, subset_questions_by_keys=None, showNotAsked=False):
    if subset_questions_by_keys is not None:
        print('---------- ASKED ----------')
    print(pickle_file)
    print('*********')
    for qa_pairs in qa_in:
        hasSub = False
        if subset_questions_by_keys is not None and type(subset_questions_by_keys) == type([]):
            for s in subset_questions_by_keys:
                if s in qa_pairs['prompt']:
                    hasSub = True
        else:
            hasSub = True

        if hasSub:
            print('Prompt:', qa_pairs['prompt'])
            print('   Real A:', qa_pairs['A'])
            print('ChatGPT A:', qa_pairs['Response'])
            print('')

    if subset_questions_by_keys is not None and showNotAsked:
        print('')
        print('')
        print('------------ NOT ASKED -----------')
        for qa_pairs in qa_in:
            hasSub = False
            if subset_questions_by_keys is not None and type(subset_questions_by_keys) == type([]):
                for s in subset_questions_by_keys:
                    if s in qa_pairs['prompt']:
                        hasSub = True

            if not hasSub:
                print('Prompt:', qa_pairs['prompt'])
                print('   Real A:', qa_pairs['A'])
                print('ChatGPT A:', qa_pairs['Response'])
                print('')

In [13]:
# setup
with open(key_file,'r') as f:
    api_key = f.read()

client = OpenAI(
  api_key=api_key.strip(),  # this is also the default, it can be omitted
)

In [14]:
jsons_to_parse = glob(jsons_dir + '/*.json')
jsons_to_parse[:3]

['/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/jsons/Picture186.json',
 '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/jsons/Picture169.json',
 '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/jsons/Picture56.json']

In [15]:
with open(redo_file,'r') as f:
    redo_q_list = json.load(f)
    redo_q_list = json.loads(redo_q_list)

questions_redo = []
for q in redo_q_list:
    questions_redo.append(q['old'])

redo_q_list, questions_redo

([{'key': 'aspect ratio',
   'old': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? You are a helpful assistant, please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.',
   'new': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.',
   'new format': 'Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.',
   'new question': 'What is the aspect ratio of this figure?',
   'new context': ''},
  {'key': 'ylabels',
   'old': 'You are a helpful assistant that can analyze images.  What are the x-axis titles for each figure panel? Please format the output as a json as {"ylabels":[]}, where the list is a list of strings of all of the y-axis titles. If there is a single plot, this should be one element in this list, and if th

In [ ]:
iMax = 2500 # should be number of jsons
verbose = False
test_run = False # run w/o actually pinging openai
restart = False

subset_questions_by_keys = None # ['median', 'ngaussians'] # set to None to do all questions

reload(utils.llm_utils)
from utils.llm_utils import parse_qa

fac = 1.0
for ijson,json_path in enumerate(jsons_to_parse):
    if ijson >= iMax:
        continue

    print('on', ijson+1, 'of', min(iMax,len(jsons_to_parse)))

    # get image and base json
    img_path = imgs_dir + json_path.split('/')[-1].removesuffix('.json') + '.' + img_format
    encoded_image, img_format_media, base_json, err = get_img_json_pair(img_path, json_path, dir_api, 
                                                      fac=fac, restart=restart,
                                                      tmp_dir=tmp_dir)

    if err:
        continue

    # also load the already-filled json from prior run
    with open(dir_api_old + json_path.split('/')[-1].removesuffix('.json') + '.pickle', 'rb') as f:
        qa_orig, model = pickle.load(f)

    # find ones to rerun
    qa = []; redos = []
    for qa_pair in qa_orig:
        if qa_pair['Q'] in questions_redo:
            Qnew = redo_q_list[questions_redo.index(qa_pair['Q'])]
            print('** REDO:', qa_pair['Q'])
            qaout = deepcopy(qa_pair)
            # empty things
            for k in ['Response', 'raw answer', 'Response String', 'prompt']:
                qaout[k] = ""
            qaout['fac'] = 1.0
            # update question
            qaout['Q'] = Qnew['new']
            qaout['format'] = Qnew['new format']
            qaout['question'] = Qnew['new question']
            qa.append(qaout)
            redos.append(True)
            print('** NEW: ', qaout)
            #import sys; sys.exit()
        else:
            qa.append(deepcopy(qa_pair))
            redos.append(False)
    #import sys; sys.exit()

    responses = []
    for question_list,rd in zip(qa,redos): # note redundent as qa gets updated anyway
        if rd:
            response, prompt = send_to_chatgpt(question_list, client, img_path, encoded_image,
                        model = model, img_format = img_format_media, temperature=temperature,
                        test_run = test_run, subset_questions_by_keys=subset_questions_by_keys, 
                        fac=fac)
            responses.append(response)
            question_list['prompt'] = prompt
        else: # just use old
            responses.append(deepcopy(question_list['Response']))

    # parse for errors
    qaout2 = parse_for_errors(deepcopy(qa))

    # dump to file
    if not test_run:
        with open(dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle', 'wb') as ff:
            pickle.dump([qaout2, model], ff)
        print("just saved:", dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
    else:
        print('Would store at:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')


    #import sys; sys.exit()

on 1 of 200
have file already: /Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/LMM_outputs/chatgpt_api/Picture186.pickle
on 2 of 200
have file already: /Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/LMM_outputs/chatgpt_api/Picture169.pickle
on 3 of 200
have file already: /Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/LMM_outputs/chatgpt_api/Picture56.pickle
on 4 of 200
have file already: /Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/LMM_outputs/chatgpt_api/Picture190.pickle
on 5 of 200
have file already: /Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/LMM_outputs/chatgpt_api/Picture40.pickle
on 6 of 200
have file already: /Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/LMM_outputs/chatgpt_api/Picture17.pickle
on 7 of 200
have file already: /Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/LMM_outputs/chatgpt_api/Picture128.pickle
on 8 of 200
have file already: /Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/LMM_outputs/chatgpt_api/Picture153.pi